# PhishGuard-VLM — Google Colab (GPU)

End-to-end setup: clone repo, CUDA PyTorch, dependencies, optional Drive, training with **LLaVA 1.5 7B + LoRA** (memory-heavy).

**Before running:** set `REPO_URL` to your fork (or upstream). Upload `data/processed/manifest.parquet` (and `data/` assets) via Drive, or run **Section 4 — Preprocessing** to build `manifest.parquet` from `data/crawl_manifest.json` after crawling.


## Section 1 — Runtime, clone, system packages


In [ ]:
# --- Edit if you use a fork ---
REPO_URL = "https://github.com/YOUR_USERNAME/Phishguard-VLM.git"
REPO_DIR = "/content/Phishguard-VLM"

import subprocess
import os

def sh(cmd_list, check=True):
    print("$", " ".join(cmd_list))
    r = subprocess.run(cmd_list, shell=False, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd_list}")

sh(["nvidia-smi"], check=False)
if os.path.isdir(REPO_DIR):
    print("Repo exists, pulling latest")
    sh(["git", "-C", REPO_DIR, "pull"])
else:
    sh(["git", "clone", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
print("CWD:", os.getcwd())

subprocess.run(["apt-get", "-qq", "-y", "update"], check=False)
subprocess.run(["apt-get", "-qq", "-y", "install", "-y", "git", "wget", "curl", "unzip"], check=False)


## Section 1b — PyTorch with CUDA (Colab)


In [ ]:
# Colab often ships torch; reinstall CUDA build matching runtime (see https://pytorch.org/get-started/locally/)
import subprocess
import sys

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# CUDA 12.4 wheels (works on many Colab GPUs); if import fails, switch index to cu121 from pytorch.org
pip_install("torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu124")

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))


**CUDA fallback:** If `torch.cuda.is_available()` is False after the cell above, re-run with `cu121` wheels:
```
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
```


In [ ]:
import torch
if not torch.cuda.is_available():
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"])
    print("Reinstalled torch with cu121. Use Runtime → Restart runtime, then re-run from Section 1b.")
else:
    print("cuda OK:", torch.cuda.get_device_name(0))


## Section 1c — Python dependencies (without reinstalling CPU torch)


In [ ]:
import subprocess, sys, os
os.chdir("/content/Phishguard-VLM")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-no-torch.txt"
])
# Playwright browsers only if you run crawl in Colab (large)
# subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=False)
print("pip deps OK")


## Section 2 — Google Drive (checkpoints + data)


In [ ]:
from google.colab import drive
import os

DRIVE_MOUNT = "/content/drive"
drive.mount(DRIVE_MOUNT)

# Folders on Drive (create if missing)
DRIVE_PG = "/content/drive/MyDrive/PhishGuard-VLM"
os.makedirs(DRIVE_PG + "/checkpoints", exist_ok=True)
os.makedirs(DRIVE_PG + "/data", exist_ok=True)
print("Drive base:", DRIVE_PG)


## Section 3 — Imports and environment checks


In [ ]:
import os
import sys

REPO_DIR = "/content/Phishguard-VLM"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
device = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Hugging Face cache on Drive (optional, avoids re-download)
os.environ.setdefault("HF_HOME", "/content/drive/MyDrive/PhishGuard-VLM/hf_cache")
os.environ.setdefault("TRANSFORMERS_CACHE", os.environ["HF_HOME"])
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# Weights & Biases off by default in Colab (set secret or env if you want logging)
os.environ.setdefault("WANDB_MODE", "disabled")

# If LLaVA weights are gated, set token: os.environ["HF_TOKEN"] = "hf_..."
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("PYTHONPATH[0]:", sys.path[0])
print("CUDA:", torch.cuda.get_device_name(0))


## Section 4 — Data preparation

Either run **Preprocessing** below to build `data/processed/manifest.parquet` from a crawl JSON (`scripts/run_crawl.py` locally or in Colab, or the repo’s sample `data/crawl_manifest.json`), or copy a ready manifest from Drive in the cell after that.


### Preprocessing — build `manifest.parquet`

Runs `scripts/run_preprocess.py` with `configs/data.yaml` (splits, image size, quality filters). Requires `data/crawl_manifest.json` paths to exist under the repo (screenshots/HTML as produced by the crawler). If you stored crawl output on Drive, this cell copies `MyDrive/PhishGuard-VLM/data/crawl_manifest.json` into the repo first.

Set `MERGE_MANIFESTS` to extra crawl JSON paths (e.g. merged phishing + benign) before running.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/Phishguard-VLM")
DRIVE_PG = Path("/content/drive/MyDrive/PhishGuard-VLM")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Optional: overwrite repo crawl manifest from Drive (after run_crawl or manual upload)
drive_crawl = DRIVE_PG / "data" / "crawl_manifest.json"
dst_crawl = REPO_DIR / "data" / "crawl_manifest.json"
if drive_crawl.is_file():
    dst_crawl.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_crawl, dst_crawl)
    print("Copied crawl manifest from Drive ->", dst_crawl)

# Extra crawl JSONs merged before preprocess (same as CLI --merge-manifest)
MERGE_MANIFESTS = []  # e.g. [REPO_DIR / "data" / "crawl_benign.json"]

if not dst_crawl.is_file():
    raise FileNotFoundError(
        f"Missing {dst_crawl}. Mount Drive and upload crawl JSON, or clone repo with sample manifest."
    )

cmd = [
    sys.executable,
    "scripts/run_preprocess.py",
    "--crawl-manifest",
    "data/crawl_manifest.json",
    "--output",
    "data/processed/manifest.parquet",
    "--config",
    "configs/data.yaml",
]
for m in MERGE_MANIFESTS:
    if m.is_file():
        cmd.extend(["--merge-manifest", str(m.relative_to(REPO_DIR))])
    else:
        print("Skip (not found):", m)

print("$", " ".join(cmd))
r = subprocess.run(cmd, cwd=str(REPO_DIR))
if r.returncode != 0:
    raise RuntimeError("run_preprocess.py failed, exit code %s" % r.returncode)

out = REPO_DIR / "data" / "processed" / "manifest.parquet"
assert out.is_file(), "Expected manifest.parquet"
import pandas as pd

print("Wrote", out, "rows:", len(pd.read_parquet(out)))

In [ ]:
import os
import shutil
from pathlib import Path

REPO_DIR = Path("/content/Phishguard-VLM")
DRIVE_PG = Path("/content/drive/MyDrive/PhishGuard-VLM")

# Skip if preprocessing already wrote manifest.parquet and you do not want Drive to overwrite it.

# Copy processed manifest + data tree from Drive if present
src_manifest = DRIVE_PG / "data" / "processed" / "manifest.parquet"
dst_manifest = REPO_DIR / "data" / "processed" / "manifest.parquet"
if src_manifest.is_file():
    dst_manifest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_manifest, dst_manifest)
    print("Copied manifest from Drive:", dst_manifest)
else:
    print("No manifest on Drive at", src_manifest)
    print("Upload manifest.parquet (+ processed images/text) there, or run preprocess locally / in Colab.")

# Optional: copy whole data directory
src_data = DRIVE_PG / "data"
if src_data.is_dir() and not (REPO_DIR / "data" / "processed").is_dir():
    shutil.copytree(src_data, REPO_DIR / "data", dirs_exist_ok=True)
    print("Copied data/ from Drive")

assert dst_manifest.is_file(), "Need data/processed/manifest.parquet — see above"
print("Rows ready for training (file exists).")


## Section 5 — Training (memory-safe defaults)


In [ ]:
import os
import subprocess
import sys

os.chdir("/content/Phishguard-VLM")

# Checkpoints saved under repo then copied to Drive in next section
ckpt_dir = "/content/Phishguard-VLM/models/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

cmd = [
    sys.executable,
    "scripts/run_train.py",
    "--manifest", "data/processed/manifest.parquet",
    "--training-config", "configs/colab_training.yaml",
    "--checkpoint-dir", ckpt_dir,
    "--use-amp",
    # CLI overrides (optional):
    # "--batch-size", "1",
    # "--epochs", "1",
    # "--eval-steps", "99999",  # skip mid-epoch eval if OOM during val
]
print("Running:", " ".join(cmd))
r = subprocess.run(cmd, cwd="/content/Phishguard-VLM")
if r.returncode != 0:
    raise RuntimeError("Training failed; exit code " + str(r.returncode))
print("Training finished OK")


### OOM troubleshooting (read before changing code)

1. **Reduce batch** — already `1` in `configs/colab_training.yaml`; keep `--use-amp`.
2. **Fewer evals** — pass `--eval-steps 99999` to validate only at epoch end (edit command above).
3. **Shorter text** — lower `text_max_length` in `run_train.py` / dataset (requires small code edit) or trim manifest.
4. **Stronger GPU** — Colab Pro (A100/L4) or smaller experiment model (not bundled; would require config swap).
5. **gradient_checkpointing** — already enabled via `configs/model.yaml` `lora.gradient_checkpointing: true`.
6. **CUDA mismatch** — reinstall torch with the wheel index that matches `nvidia-smi` driver (try `cu121` vs `cu124` from pytorch.org).


## Section 6 — Save checkpoints to Drive


In [ ]:
import shutil
from pathlib import Path

repo_ckpt = Path("/content/Phishguard-VLM/models/checkpoints")
drive_ckpt = Path("/content/drive/MyDrive/PhishGuard-VLM/checkpoints")
drive_ckpt.mkdir(parents=True, exist_ok=True)

if repo_ckpt.is_dir():
    for f in repo_ckpt.glob("*.pt"):
        dest = drive_ckpt / f.name
        shutil.copy2(f, dest)
        print("Saved:", dest)
else:
    print("No checkpoints dir")


## Optional — quick GPU sanity (forward one batch)


In [ ]:
# Skip if training already ran successfully
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
